# Pipeline reproducible y calidad de datos

**Proyecto Final Henry · Dynamo Consultoría & Data Solutions**

Este notebook muestra cómo el proyecto pasa de los CSV crudos de Olist a la comparación de modelos con un solo comando, y documenta la calidad de los datos y las decisiones tomadas.

1. **Configuración**
2. **El pipeline:** qué hace cada paso
3. **Los datos:** la tabla de interacciones
4. **Calidad de datos:** faltantes, outliers y desbalance
5. **Features**
6. **Modelos:** comparación de resultados
7. **Reproducibilidad**
8. **Limitaciones e ideas**
9. **Conclusiones**

**Cómo usarlo:** por defecto lee los resultados guardados en `reports/`, así se ejecuta en segundos. Para recalcular todo desde cero, cambiar `EJECUTAR_PIPELINE = True` en la sección 2 (~7 minutos).


## 1. Configuración


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import INTERACTIONS_PATH, REPORTS_DIR

pd.set_option("display.width", 140)
AZUL, GRIS = "#2a78d6", "#b9b8b2"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.edgecolor": "#888"})
print("Raíz del proyecto:", ROOT)

Raíz del proyecto: c:\Users\denny\proyectos\dynamo-recsys-olist


## 2. El pipeline: qué hace cada paso

Todo el proyecto se ejecuta con un solo comando, `python -m src.pipeline`, que corre estos 5 pasos en orden. Cada paso usa lo que produjo el anterior, por eso el orden importa.

| Paso | Script | Recibe | Produce |
|---|---|---|---|
| 1. Datos crudos | `src/data/download_data.py` | — | 9 CSV de Olist en `data/raw/` |
| 2. Interacciones | `src/data/make_interactions.py` | Los 9 CSV | `data/processed/interactions.parquet` |
| 3. Calidad de datos | `src/data/data_quality.py` | `interactions.parquet` + clientes | `reports/dq_*.csv` |
| 4. Features | `src/data/feature_engineering.py` | `interactions.parquet` | `user_features.parquet`, `product_features.parquet` |
| 5. Modelos | `src/models/train_compare.py` | `interactions.parquet` | `reports/model_comparison.csv` |

**Por qué un script `.py` y no todo en el notebook:** el `.py` se puede automatizar (GitHub Actions, MLflow) y lo pueden importar la API y la demo. Este notebook solo lo ejecuta y muestra sus resultados.

**Dos decisiones de diseño:**
- La calidad de datos (paso 3) va **después** de las interacciones (paso 2), porque necesita `interactions.parquet`.
- Si un script todavía no existe en la rama, el pipeline lo marca como **saltado** y sigue, en lugar de romperse.

In [3]:
# Interruptor: False = usar resultados guardados (segundos) | True = recalcular todo (~7 min)
from datetime import datetime

EJECUTAR_PIPELINE = False

if EJECUTAR_PIPELINE:
    from src import pipeline
    pipeline.main()
else:
    print("Usando los resultados guardados en reports/. Archivos disponibles:\n")
    for archivo in sorted(REPORTS_DIR.glob("*.csv")):
        fecha = datetime.fromtimestamp(archivo.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
        print(f"  {archivo.name:<32} actualizado: {fecha}")

Usando los resultados guardados en reports/. Archivos disponibles:

  baseline_results.csv             actualizado: 2026-10-08 16:33
  dq_cantidades.csv                actualizado: 2026-10-08 16:54
  dq_desbalance.csv                actualizado: 2026-10-08 16:54
  dq_faltantes.csv                 actualizado: 2026-10-08 16:54
  dq_nulos_disfrazados.csv         actualizado: 2026-10-08 16:54
  dq_outliers.csv                  actualizado: 2026-10-08 16:54
  model_comparison.csv             actualizado: 2026-10-08 17:00
  validation_recent_window.csv     actualizado: 2026-10-08 16:56
